# Phase 4 — Part A Diagnostic (test_year=2020 only)

Quick truncated ablation: 4 conditions on one LOYO fold (~30 min total).

The question this notebook answers:

> **Does the phase-resolved attention mechanism add real value over uniform
> mean pooling, or is the Phase 4 improvement just an ensemble effect?**

The four conditions:

| Condition | Attention | Train data |
|---|---|---|
| `MIL_resolved` | phase-resolved (per-phase MLP) | all 10 fields per bag |
| `MIL_shared`   | phase-shared (one MLP for all phases) | all 10 fields per bag |
| `mean_pool`    | uniform 1/10 (no learning) | all 10 fields per bag |
| `rank1`        | n/a (1 field per bag) | only closest field |

We compare station-level macro MAE across the four. If `mean_pool` ≈ `MIL_resolved`,
attention is not doing real work.


## 1. Setup

In [1]:
from pathlib import Path
import sys, os, time, gc, psutil
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
OUTPUT_DIR  = PROJECT_ROOT / 'phase4_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS       = [2017, 2018, 2019, 2020, 2021]
TEST_YEAR   = 2020
EPOCHS      = 15
HIDDEN      = 32
BATCH_SIZE  = 8
LR          = 1e-3
USE_AMP     = True

def memstat(tag=''):
    cpu = psutil.Process(os.getpid()).memory_info().rss / 1e9
    if torch.cuda.is_available():
        return f'{tag} RAM={cpu:.2f}GB  GPU={torch.cuda.memory_allocated()/1e9:.2f}GB'
    return f'{tag} RAM={cpu:.2f}GB'
print(memstat('init:'))


Device: cuda
init: RAM=0.40GB  GPU=0.00GB


## 2. Helpers

In [2]:
from phase3_baselines.dataset import (load_year_bundle, SelectionPolicy,
                                       apply_selection, concat_bundles)
from phase3_baselines.evaluate import evaluate_predictions
from phase4_mil.bag_dataset import build_bags
from phase4_mil.train import train_mil_fold, STREAM_CONFIGS
from phase4_mil import model as _model_mod
from phase4_mil import train as _train_mod
from phase4_mil.model import MILModel


def load_loyo_bags(test_year, use_rank1=False):
    policy = SelectionPolicy(use_only_rank1=use_rank1)
    test_b = apply_selection(
        load_year_bundle(TENSORS_DIR / f'phenology_{test_year}.npz'), policy)
    train_parts = {}
    for y in YEARS:
        if y == test_year: continue
        train_parts[y] = apply_selection(
            load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
    train_b = concat_bundles(train_parts, years=[y for y in YEARS if y != test_year])
    del train_parts; gc.collect()
    return build_bags(train_b), build_bags(test_b)


class MeanPoolModel(MILModel):
    """MILModel with forced uniform attention. We still keep the scorer params
    in the module (unused), so checkpointing/printing don't break."""
    def forward(self, batch, active=('S1','S2','Met')):
        X = batch['X']; fmask = batch['feat_mask']; bmask = batch['bag_mask']
        h_seq = self.encode_bag(X, fmask, active=active)
        bm = bmask.float()
        counts = bm.sum(dim=1, keepdim=True).clamp(min=1.0)
        alpha_unif = (bm / counts).unsqueeze(-1).expand(-1, -1, self.n_phases)
        z = torch.einsum('bjth,bjp->btph', h_seq, alpha_unif)
        station_logits = torch.einsum('btph,ph->btp', z, self.station_head) + self.station_bias
        field_logits = self.field_head(h_seq)
        return {'station_logits': station_logits,
                'field_logits':   field_logits,
                'alpha':          alpha_unif,
                'h_field':        h_seq.mean(dim=2)}


## 3. Pre-load bags for the test_year=2020 fold

In [3]:
print('Loading bags (all10)...')
train_bags_a10, test_bags_a10 = load_loyo_bags(TEST_YEAR, use_rank1=False)
print(f'  train: {train_bags_a10["X"].shape}, test: {test_bags_a10["X"].shape}')

print('Loading bags (rank1)...')
train_bags_r1, test_bags_r1 = load_loyo_bags(TEST_YEAR, use_rank1=True)
print(f'  train: {train_bags_r1["X"].shape}, test: {test_bags_r1["X"].shape}')

print(memstat('after load:'))
phase_ids = train_bags_a10['phase_ids']


Loading bags (all10)...
  train: (2424, 10, 426, 19), test: (585, 10, 426, 19)
Loading bags (rank1)...
  train: (2421, 10, 426, 19), test: (585, 10, 426, 19)
after load: RAM=2.92GB  GPU=0.00GB


## 4. Run the four conditions

In [4]:
results_rows = []
all_alphas = {}

def evaluate(result, condition):
    rep_st = evaluate_predictions(
        pred_days=result['station_pred'].astype(np.float32),
        true_days=result['station_truth'].astype(np.float32),
        phase_ids=phase_ids, years=result['station_year'], ranks=None,
    )
    rep_fd = evaluate_predictions(
        pred_days=result['field_pred'].astype(np.float32),
        true_days=result['field_truth'].astype(np.float32),
        phase_ids=phase_ids, years=result['field_year'],
        ranks=result['field_rank'],
    )
    for phase, m in rep_st['per_phase'].iterrows():
        results_rows.append({'condition': condition, 'level': 'station',
                              'phase': int(phase), **m.to_dict()})
    for phase, m in rep_fd['per_phase'].iterrows():
        results_rows.append({'condition': condition, 'level': 'field',
                              'phase': int(phase), **m.to_dict()})

# ---- 4a. MIL_resolved (phase-resolved attention) ----
print('=== MIL_resolved ===')
t0 = time.time()
res = train_mil_fold(
    train_bags=train_bags_a10, test_bags=test_bags_a10,
    active=STREAM_CONFIGS['All'],
    hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    phase_resolved_attention=True, field_loss_weight=0.3,
    use_amp=USE_AMP, device=DEVICE, verbose=False,
)
print(f'  done in {time.time()-t0:.0f}s')
evaluate(res, 'MIL_resolved')
all_alphas['MIL_resolved'] = res['alpha']
print(memstat())


=== MIL_resolved ===
  done in 158s
 RAM=3.68GB  GPU=0.02GB


In [5]:
# ---- 4b. MIL_shared (phase-shared attention) ----
print('=== MIL_shared ===')
t0 = time.time()
res = train_mil_fold(
    train_bags=train_bags_a10, test_bags=test_bags_a10,
    active=STREAM_CONFIGS['All'],
    hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    phase_resolved_attention=False, field_loss_weight=0.3,
    use_amp=USE_AMP, device=DEVICE, verbose=False,
)
print(f'  done in {time.time()-t0:.0f}s')
evaluate(res, 'MIL_shared')
all_alphas['MIL_shared'] = res['alpha']
print(memstat())


=== MIL_shared ===
  done in 146s
 RAM=3.68GB  GPU=0.02GB


In [6]:
# ---- 4c. mean_pool (uniform attention) ----
print('=== mean_pool ===')
t0 = time.time()
# Monkey-patch the model symbol used in train_mil_fold
_orig_model = _model_mod.MILModel
_orig_train = _train_mod.MILModel
_model_mod.MILModel = MeanPoolModel
_train_mod.MILModel = MeanPoolModel
try:
    res = train_mil_fold(
        train_bags=train_bags_a10, test_bags=test_bags_a10,
        active=STREAM_CONFIGS['All'],
        hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
        phase_resolved_attention=False, field_loss_weight=0.3,
        use_amp=USE_AMP, device=DEVICE, verbose=False,
    )
finally:
    _model_mod.MILModel = _orig_model
    _train_mod.MILModel = _orig_train
print(f'  done in {time.time()-t0:.0f}s')
evaluate(res, 'mean_pool')
all_alphas['mean_pool'] = res['alpha']
print(memstat())


=== mean_pool ===
  done in 144s
 RAM=3.68GB  GPU=0.02GB


In [7]:
# ---- 4d. rank1 (closest field only — no MIL since each bag has 1 field) ----
print('=== rank1 ===')
t0 = time.time()
res = train_mil_fold(
    train_bags=train_bags_r1, test_bags=test_bags_r1,
    active=STREAM_CONFIGS['All'],
    hidden=HIDDEN, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    phase_resolved_attention=False, field_loss_weight=0.3,
    use_amp=USE_AMP, device=DEVICE, verbose=False,
)
print(f'  done in {time.time()-t0:.0f}s')
evaluate(res, 'rank1')
all_alphas['rank1'] = res['alpha']
print(memstat())


=== rank1 ===
  done in 143s
 RAM=3.68GB  GPU=0.02GB


## 5. Summary

In [8]:
results = pd.DataFrame(results_rows)

# Headline: macro MAE per (condition, level)
print('=== HEADLINE: macro MAE per condition (mean over phases) ===')
summary = (results.groupby(['condition','level'])
                   .agg(macro_mae=('mae','mean'),
                        macro_r2 =('r2','mean'),
                        macro_within_5d=('within_5d','mean'))
                   .reset_index())
print(summary.round(3).to_string(index=False))


=== HEADLINE: macro MAE per condition (mean over phases) ===
   condition   level  macro_mae  macro_r2  macro_within_5d
MIL_resolved   field      9.979    -0.412            0.373
MIL_resolved station      9.107    -0.194            0.404
  MIL_shared   field      9.363    -0.283            0.391
  MIL_shared station      8.660    -0.250            0.420
   mean_pool   field      9.500    -0.299            0.382
   mean_pool station      8.752    -0.110            0.415
       rank1   field      8.730    -0.139            0.401
       rank1 station      8.558    -0.097            0.411


In [9]:
# Per-phase MAE — station level
station = results[results['level']=='station']
table_st = station.pivot(index='phase', columns='condition', values='mae').round(2)
print('\n=== Station-level MAE by (condition, phase) ===')
print(table_st.to_string())

# Per-phase MAE — field level
field = results[results['level']=='field']
table_fd = field.pivot(index='phase', columns='condition', values='mae').round(2)
print('\n=== Field-level MAE by (condition, phase) ===')
print(table_fd.to_string())



=== Station-level MAE by (condition, phase) ===
condition  MIL_resolved  MIL_shared  mean_pool  rank1
phase                                                
10                15.19       12.07      12.91  10.09
12                11.75       10.90      11.80  10.46
15                 8.90       10.02       9.12   8.88
18                 8.11        8.16       7.83   9.45
19                 7.72        7.59       7.77   8.27
21                 7.05        6.94       6.75   7.20
24                 5.03        4.94       5.08   5.56

=== Field-level MAE by (condition, phase) ===
condition  MIL_resolved  MIL_shared  mean_pool  rank1
phase                                                
10                16.74       14.57      14.53  10.11
12                13.77       11.65      12.96  10.24
15                10.98       10.79      10.28   9.61
18                 7.06        7.39       7.53   9.85
19                 8.03        7.99       8.10   8.40
21                 7.44        7.39     

## 6. Attention diagnostics

In [10]:
# Compare attention patterns across the three pooling conditions
# (rank1 alpha is degenerate so we skip it here)
print('=== Coefficient of variation across slots, per (condition, phase) ===')
print('(higher = attention is more selective; uniform = 0)')
print()
for cond in ['MIL_resolved', 'MIL_shared', 'mean_pool']:
    alpha = all_alphas[cond]                     # (N_bags, 10, P)
    mean_alpha = alpha.mean(axis=0)              # (10, P)
    cv = mean_alpha.std(axis=0) / (mean_alpha.mean(axis=0) + 1e-8)
    cv_str = '  '.join(f'p{p}:{c:.3f}' for p, c in zip(phase_ids, cv))
    print(f'  {cond:14s}: {cv_str}')

print()
print('=== Mean attention by (slot, phase) — MIL_resolved ===')
mr = all_alphas['MIL_resolved'].mean(axis=0)
df = pd.DataFrame(mr.round(3),
                    index=[f'slot{i+1}' for i in range(10)],
                    columns=[f'p{p}' for p in phase_ids])
print(df.to_string())

print()
print('=== Mean attention by (slot, phase) — MIL_shared ===')
ms = all_alphas['MIL_shared'].mean(axis=0)
df_s = pd.DataFrame(ms.round(3),
                    index=[f'slot{i+1}' for i in range(10)],
                    columns=[f'p{p}' for p in phase_ids])
print(df_s.to_string())


=== Coefficient of variation across slots, per (condition, phase) ===
(higher = attention is more selective; uniform = 0)

  MIL_resolved  : p10:0.198  p12:0.204  p15:0.210  p18:0.210  p19:0.212  p21:0.212  p24:0.217
  MIL_shared    : p10:0.205  p12:0.205  p15:0.205  p18:0.205  p19:0.205  p21:0.205  p24:0.205
  mean_pool     : p10:0.210  p12:0.210  p15:0.210  p18:0.210  p19:0.210  p21:0.210  p24:0.210

=== Mean attention by (slot, phase) — MIL_resolved ===
          p10    p12    p15    p18    p19    p21    p24
slot1   0.142  0.144  0.146  0.146  0.146  0.146  0.148
slot2   0.121  0.123  0.125  0.125  0.126  0.126  0.126
slot3   0.118  0.116  0.113  0.113  0.112  0.112  0.110
slot4   0.102  0.103  0.103  0.103  0.103  0.103  0.104
slot5   0.096  0.096  0.095  0.095  0.095  0.095  0.095
slot6   0.090  0.090  0.091  0.091  0.092  0.092  0.093
slot7   0.089  0.087  0.086  0.086  0.086  0.086  0.086
slot8   0.083  0.083  0.083  0.083  0.083  0.083  0.083
slot9   0.081  0.081  0.080  0.080 

In [11]:
"""
Add-on cell for the truncated Part A diagnostic notebook (04b).

Saves the attention weights and predictions to phase4_outputs/mil_partA_predictions.npz
in the format that fig5_attention_heatmap expects.

If you have already closed the notebook session, just rerun 04b end-to-end and
then run this cell at the end. Total time ~30 min if the four training fits
have to be redone; ~5 seconds if the notebook is still in memory and
`all_alphas` dict from Section 6 is still alive.

After running, regenerate F5 via notebook 09:
    from figures.style import apply_style
    from figures.figures import fig5_attention_heatmap
    apply_style()
    fig5_attention_heatmap(PROJECT_ROOT, PROJECT_ROOT / 'phase6_figures')
"""

import numpy as np
from pathlib import Path
import sys
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'phase4_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NPZ_PATH = OUTPUT_DIR / 'mil_partA_predictions.npz'

# all_alphas is a dict mapping condition_name -> (N_bags, 10, 7) array of attention weights.
# It is created in Section 6 of notebook 04b. If you've closed the notebook,
# rerun 04b first so `all_alphas` is back in memory.

if 'all_alphas' not in dir():
    raise RuntimeError(
        "Variable `all_alphas` not found. Run the cells in 04b that build it "
        "(Sections 4a, 4b, 4c, 4d) before running this saver cell."
    )

# Save in the key format expected by fig5: '{condition}__All__{year}__alpha'
# Since the truncated diagnostic only used test_year=2020, we save under that year.
TEST_YEAR = 2020

arrays = {}
for cond, alpha in all_alphas.items():
    if alpha is None:
        continue
    key = f'{cond}__All__{TEST_YEAR}__alpha'
    arrays[key] = alpha.astype(np.float32)
    print(f'  {key}: shape={alpha.shape}')

# If a previous file exists, merge with it rather than overwrite
existing = {}
if NPZ_PATH.exists():
    with np.load(NPZ_PATH, allow_pickle=False) as z:
        existing = {k: z[k] for k in z.files}
    print(f'Merging with {len(existing)} existing arrays in {NPZ_PATH.name}')

existing.update(arrays)
np.savez_compressed(NPZ_PATH, **existing)
print(f'\nSaved {len(arrays)} attention arrays to {NPZ_PATH}')
print(f'Total arrays in file: {len(existing)}')
print('\nNow re-run figure F5 generation via notebook 09.')

  MIL_resolved__All__2020__alpha: shape=(585, 10, 7)
  MIL_shared__All__2020__alpha: shape=(585, 10, 7)
  mean_pool__All__2020__alpha: shape=(585, 10, 7)
  rank1__All__2020__alpha: shape=(585, 10, 7)

Saved 4 attention arrays to /fs1/home/atikul/Phenology/phase4_outputs/mil_partA_predictions.npz
Total arrays in file: 4

Now re-run figure F5 generation via notebook 09.


## 7. What to look for

**The most informative comparison is `MIL_resolved` vs `mean_pool`** at the station level.

| If we see... | We conclude... | What's next |
|---|---|---|
| `mean_pool` ≈ `MIL_resolved` (within ±0.3 days) | Attention is doing nothing; gain is from ensemble pooling | Try entropy penalty or better scorer input (Path B fix) |
| `MIL_resolved` clearly beats `mean_pool` (>0.5 days) | Attention adds real value | Proceed to full grid as designed |
| `MIL_shared` ≈ `MIL_resolved` | Phase-specific attention adds nothing over a shared learned weight | Simplify to phase-shared in final paper |

Also check `rank1` vs the pooling conditions: if `rank1` is much worse despite using only the
"closest" field, that confirms the value of multi-field training data.
